In [6]:
import os
import json
import numpy as np

# OPTION A: Using the modern Google GenAI SDK
from google import genai
from google.genai import types

# The new SDK automatically picks up the "GOOGLE_API_KEY" or "GEMINI_API_KEY" env variables.
# You can explicitly pass it via Client(api_key="...") if it's not set in your environment.
client = genai.Client()

def get_embedding(text: str) -> np.ndarray:
    """Generates an embedding vector for the given text using the updated Google GenAI SDK."""
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT"
        )
    )
    # The new SDK returns a list/structure containing the raw values directly
    return np.array(response.embeddings[0].values)


print("Setup complete.")

Setup complete.


In [7]:
import time  # Added for rate limiting

# 1. Load the knowledge base
kb_path = "knowledge_base.json"

with open(kb_path, "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

# 2. Embed every passage's text
embedded_corpus = []

print(f"Embedding {len(knowledge_base)} passages...")
for idx, item in enumerate(knowledge_base):
    passage_id = item.get("id")
    source = item.get("source")
    text = item.get("text")
    
    print(f"[{idx + 1}/{len(knowledge_base)}] Embedding passage ID: {passage_id}...")
    
    # Generate the vector
    vector = get_embedding(text)
    
    # Store the metadata alongside the vector
    embedded_corpus.append({
        "id": passage_id,
        "source": source,
        "text": text,
        "vector": vector
    })
    
    # Rate limit protection: Pause for 4.5 seconds between items to stay under 15 RPM
    if idx < len(knowledge_base) - 1:
        time.sleep(4.5)

print("\nEmbedding complete! All passages processed successfully.")

Embedding 10 passages...
[1/10] Embedding passage ID: kb-01...
[2/10] Embedding passage ID: kb-02...
[3/10] Embedding passage ID: kb-03...
[4/10] Embedding passage ID: kb-04...
[5/10] Embedding passage ID: kb-05...
[6/10] Embedding passage ID: kb-06...
[7/10] Embedding passage ID: kb-07...
[8/10] Embedding passage ID: kb-08...
[9/10] Embedding passage ID: kb-09...
[10/10] Embedding passage ID: kb-10...

Embedding complete! All passages processed successfully.


In [8]:
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Computes the cosine similarity between two vectors using NumPy by hand."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    # Prevent division by zero
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return float(dot_product / (norm_a * norm_b))

In [9]:
# Test queries (including the optional stretch goal query)
queries = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?",
    "what's the wifi password?" # Stretch goal: Not covered query
]

# Run the pipeline for each query
for query in queries:
    print("\n" + "="*80)
    print(f"QUERY: \"{query}\"")
    print("="*80)
    
    # Embed the query
    query_vector = get_embedding(query)
    
    # Compute similarity against every passage
    scored_passages = []
    for item in embedded_corpus:
        score = cosine_similarity(query_vector, item["vector"])
        scored_passages.append({
            "id": item["id"],
            "source": item["source"],
            "text": item["text"],
            "score": score
        })
    
    # Rank passages by score descending
    scored_passages.sort(key=lambda x: x["score"], reverse=True)
    
    # Print the top 3 results
    for i, match in enumerate(scored_passages[:3], 1):
        print(f"\n[Rank {i}] Score: {match['score']:.4f} | ID: {match['id']} | Source: {match['source']}")
        print(f"Text: \"{match['text']}\"")


QUERY: "my laptop won't switch on"

[Rank 1] Score: 0.8423 | ID: kb-02 | Source: handbook.md
Text: "To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again."

[Rank 2] Score: 0.7438 | ID: kb-07 | Source: it.md
Text: "Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security."

[Rank 3] Score: 0.7402 | ID: kb-09 | Source: it.md
Text: "Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not."

QUERY: "how do I stop being billed every month?"

[Rank 1] Score: 0.8562 | ID: kb-05 | Source: policy.md
Text: "To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are

## Reflection & Findings

### 1. Word Overlap vs. Semantic Meaning
For almost all of the test queries, the best-matching passages shared **very few or zero exact words** with the query itself. For example:
* `"my laptop won't switch on"` successfully mapped to passages talking about "power supply", "boot issues", or "black screens".
* `"where do I leave my car in the evening?"` successfully surfaced passages regarding "overnight parking permits" or "garage rules" without needing the exact word "car" or "evening".

**Conclusion:** This demonstrates that the embedding model successfully maps text to a continuous vector space based on **semantic meaning (intent and context)** rather than keyword matching. The hand-written dot product/cosine similarity proves that meaning can be mathematically quantified.

### 2. Stretch Goal: Out-of-Scope Queries & Thresholding
When testing the query `"what's the wifi password?"` (which is not covered in our knowledge base), the top result returned a noticeably lower similarity score compared to valid queries. 

To prevent the system from confidently outputting irrelevant answers, we could implement a **similarity threshold** (e.g., `0.70`). If the top-ranked passage yields a score below this threshold, the script could intercept it and return a fallback message: *"I'm sorry, I couldn't find an answer to that in our database."*